# Updating the Document or add the new docs into existing vector store

In [42]:
import os
from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from nltk.data import retrieve
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
# from data_ingest_parsing.data_ingestion import documents

load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv('OPENAI_API_KEY')
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')


from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_core.documents import Document

## vector store
from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List



In [2]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn
    and improve from experience without being explicitly programmed. There are three main
    types of machine learning: supervised learning, unsupervised learning, and reinforcement
    learning. Supervised learning uses labeled data to train models, while unsupervised
    learning finds patterns in unlabeled data. Reinforcement learning learns through
    interaction with an environment using rewards and penalties.
    """,

    """
    Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks.
    These networks are inspired by the human brain and consist of layers of interconnected
    nodes. Deep learning has revolutionized fields like computer vision, natural language
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers
    excel at sequential data processing.
    """,

    """
    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language.
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis,
    machine translation, and question answering. Modern NLP heavily relies on transformer
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn\n    and improve from experience without being explicitly programmed. There are three main\n    types of machine learning: supervised learning, unsupervised learning, and reinforcement\n    learning. Supervised learning uses labeled data to train models, while unsupervised\n    learning finds patterns in unlabeled data. Reinforcement learning learns through\n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks.\n    These networks are inspired by the human brain and consist of layers of interconnected\n    nodes. Deep learning has revolutionized fields like computer vision, natural language\n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly\n    effective for image 

In [3]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. ...


In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from 3 documents

Chunk example:
Content: Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from exp...
Metadata: {'source': 'data\\doc_0.txt'}


In [5]:
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Vector store created with 40 vectors
Persisted to: ./chroma_db


In [6]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make
decisions by interacting with an environment. The agent receives rewards or penalties
based on its actions and learns to maximize cumulative reward over time. Key concepts
in RL include: states, actions, rewards, policies, and value functions. Popular RL
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),
robotics, and autonomous systems.
"""

In [7]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning Fundamentals\n    \n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep Learning and Neural Networks\n    \n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of laye

In [8]:
new_doc = Document(
    page_content=new_document,
    metadata={"source":"manual_addition", "topic":"RL"}
)

In [9]:
new_doc

Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.\n')

In [10]:
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make\ndecisions by interacting with an environment. The agent receives rewards or penalties\nbased on its actions and learns to maximize cumulative reward over time. Key concepts\nin RL include: states, actions, rewards, policies, and value functions. Popular RL\nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and\nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_addition', 'topic': 'RL'}, page_content='methods, and\nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo),\nrobotics, and autonomous systems.')]

In [11]:
vectorstore.add_documents(new_chunks)

['93dce6b1-bf65-46ea-8072-2b6fd5f4ec5e',
 'b17b66f4-7755-4718-b5a3-86938cd2eade']

In [12]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 2 new chunks to the vector store
Total vectors now: 42


In [13]:
# pip install langchain-core langchain-openai

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# === you already have this ===
# documents: list[Document] from DirectoryLoader

def format_docs(docs):
    return "\n\n".join(getattr(d, "page_content", str(d)) for d in docs) if docs else "No relevant context found."

SYSTEM = """You are an assistant for question-answer tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences max and keep the answer concise.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ✅ Pass the docs at invoke-time
chain = (
    RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke({
    "input": "what is deep learning?",
    "context": documents   # <-- your loaded docs go here
})

print(answer)


Deep learning is a subset of machine learning that is based on artificial neural networks, which are inspired by the human brain. It involves layers of interconnected nodes and has significantly advanced fields such as computer vision, natural language processing, and speech recognition. Deep learning techniques, like Convolutional Neural Networks (CNNs) and Recurrent Neural Networks (RNNs), are particularly effective for processing images and sequential data, respectively.


In [14]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000270435F2950>, search_kwargs={})

In [15]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs) if docs else "No relevant context found."

In [16]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [17]:
rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [18]:
def query_rag_lcel(question):
    print(f"question: {question}")
    print("-"*50)

    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    source = retriever.invoke(answer)
    for s in source:
        print(s.page_content)

In [19]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

question: What are the keys concepts in reinforcement learning
--------------------------------------------------
Answer: The key concepts in reinforcement learning (RL) include:

1. **States**: These represent the different situations or configurations that the agent can encounter in the environment.
2. **Actions**: These are the choices or moves that the agent can make in response to the current state.
3. **Rewards**: These are the feedback signals received by the agent based on the actions it takes, which can be positive (rewards) or negative (penalties).
4. **Policies**: These define the strategy that the agent employs to determine its actions based on the current state.
5. **Value Functions**: These are used to estimate the expected cumulative reward that can be obtained from a given state or state-action pair.

These concepts are fundamental to how an agent learns to make decisions in reinforcement learning.
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type 

# Advanced Rag Techniques -  Conversational Memory

In [20]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [21]:
## create the prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

In [22]:
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000270435F2950>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIM

In [23]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)


In [24]:
chat_history=[]
# First question
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})

In [25]:
print(f"Q: What is machine learning?")
print(f"A: {result1['answer']}")

Q: What is machine learning?
A: Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves three main types: supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with the environment.


In [26]:
print(chat_history)

[]


In [27]:
result1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are they?"
})

In [28]:
print(chat_history)

[]


In [29]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=result1['answer'])
])

In [30]:
result2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"  # Refers to ML from previous question
})

In [31]:
result2['answer']

'The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with an environment to maximize rewards.'

In [32]:
chat_history.extend([
    HumanMessage(content="What are its main types"),
    AIMessage(content=result2['answer'])
])

In [33]:
print(chat_history)

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}), AIMessage(content='Recurrent Neural Networks (RNNs) and Transformers are types of neural network architectures used for processing sequential data. RNNs are designed to handle sequences by maintaining a hidden state that captures information from previous inputs, while Transformers use self-attention mechanisms to process sequences in parallel, improving efficiency and performance. Both are widely used in tasks such as natural language processing and time series analysis.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What are its main types', additional_kwargs={}, response_metadata={}), AIMessage(content='The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinf

In [34]:
result3 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "how are they diff from rnn?"  # Refers to ML from previous question
})

print(result3['answer'])

RNNs are a specific type of neural network architecture designed for processing sequential data, while the main types of machine learning (supervised, unsupervised, and reinforcement learning) refer to different learning paradigms. RNNs can be used within supervised or unsupervised learning frameworks, particularly for tasks involving sequences, such as time series or natural language. In contrast, the main types of machine learning describe the overall approach to training models rather than a specific architecture.


In [35]:
def conversational_pipeline(query,chat_history):
    results = conversational_rag_chain.invoke({
        "chat_history": chat_history,
        "input": query
    })
    chat_history.extend([
        HumanMessage(content=query),
        AIMessage(content=results['answer'])
    ])
    return results["answer"]

In [36]:
conversational_pipeline("ok so what are the types then", chat_history)

'The types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data, unsupervised learning finds patterns in unlabeled data, and reinforcement learning learns through interactions with an environment.'

In [37]:
conversational_pipeline("ok is rl also under ml?", chat_history)

'Yes, reinforcement learning (RL) is a type of machine learning (ML). It involves an agent learning to make decisions by interacting with an environment to maximize cumulative rewards.'

In [38]:
print(chat_history)

[HumanMessage(content='What is machine learning', additional_kwargs={}, response_metadata={}), AIMessage(content='Recurrent Neural Networks (RNNs) and Transformers are types of neural network architectures used for processing sequential data. RNNs are designed to handle sequences by maintaining a hidden state that captures information from previous inputs, while Transformers use self-attention mechanisms to process sequences in parallel, improving efficiency and performance. Both are widely used in tasks such as natural language processing and time series analysis.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What are its main types', additional_kwargs={}, response_metadata={}), AIMessage(content='The main types of machine learning are supervised learning, unsupervised learning, and reinforcement learning. Supervised learning uses labeled data to train models, unsupervised learning finds patterns in unlabeled data, and reinf

In [39]:
conversational_pipeline("what i was taking about ?", chat_history)

'You were asking about the types of machine learning, specifically whether reinforcement learning is included under machine learning.'

### groq

In [40]:
llm

ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000027045F4D2D0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000270474A99F0>, root_client=<openai.OpenAI object at 0x0000027045F4E620>, root_async_client=<openai.AsyncOpenAI object at 0x00000270474A8070>, model_name='gpt-4o-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

In [43]:
groq_llm = ChatGroq(
    model = "gemma2-9b-it"
)

In [44]:
groq_llm

ChatGroq(profile={'max_input_tokens': 8192, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027047261F00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000270472625F0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))